# BSH Sequence Alignment & Conservation Analysis

This notebook performs multiple sequence alignment (MSA) of 127 BSH (bile salt hydrolase) enzyme sequences and analyzes positional conservation across the family.

**Workflow:**
1. Load sequences from FASTA
2. Run MUSCLE multiple sequence alignment
3. Compute per-position conservation scores
4. Identify conserved vs non-conserved positions
5. Map non-conserved alignment positions back to original residue indices
6. Export results for downstream per-residue embedding extraction

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
import os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent  # assumes running from notebooks/
FASTA_FILE = PROJECT_ROOT / "data" / "Seqs_list_total.fasta"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

MUSCLE_BIN = PROJECT_ROOT / "muscle"  # compiled MUSCLE v3 binary
ALIGNED_FASTA = OUTPUT_DIR / "bsh_aligned.fasta"

# Conservation threshold: position is "conserved" if >= this fraction
# of sequences share the same residue (excluding gaps)
CONSERVATION_THRESHOLD = 0.95

# Positions with gap frequency above this are flagged as high-gap indel positions
GAP_THRESHOLD = 0.50

print(f"FASTA file: {FASTA_FILE}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"MUSCLE binary: {MUSCLE_BIN}")
print(f"Conservation threshold: {CONSERVATION_THRESHOLD}")
print(f"Gap threshold: {GAP_THRESHOLD}")

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import subprocess
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio import SeqIO, AlignIO
from Bio.Align import MultipleSeqAlignment

In [ ]:
# ── Load FASTA & summary stats ──────────────────────────────────────────────
records = list(SeqIO.parse(FASTA_FILE, "fasta"))
print(f"Loaded {len(records)} sequences")

# Extract enzyme IDs (last token of header, e.g. 'A0A0P0LK75' from 'cluster4_A0A0P0LK75')
enzyme_ids = [rec.id.split("_")[-1] for rec in records]
lengths = [len(rec.seq) for rec in records]

print(f"\nLength distribution:")
print(f"  Min: {min(lengths)} aa")
print(f"  Max: {max(lengths)} aa")
print(f"  Mean: {np.mean(lengths):.0f} aa")
print(f"  Median: {np.median(lengths):.0f} aa")

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(lengths, bins=30, edgecolor="black", alpha=0.7)
ax.set_xlabel("Sequence length (aa)")
ax.set_ylabel("Count")
ax.set_title("BSH sequence length distribution")
plt.tight_layout()
plt.show()

print(f"\nFirst 5 IDs: {enzyme_ids[:5]}")

In [ ]:
# ── Run MUSCLE alignment ────────────────────────────────────────────────────
# MUSCLE v3 command: muscle -in <input> -out <output>
cmd = [str(MUSCLE_BIN), "-in", str(FASTA_FILE), "-out", str(ALIGNED_FASTA)]
print(f"Running: {' '.join(cmd)}")

result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print(f"MUSCLE stderr:\n{result.stderr}")
    raise RuntimeError(f"MUSCLE failed with return code {result.returncode}")

print(f"\nAlignment saved to: {ALIGNED_FASTA}")
print(f"File size: {ALIGNED_FASTA.stat().st_size / 1024:.1f} KB")

# Parse alignment
alignment = AlignIO.read(ALIGNED_FASTA, "fasta")
n_seqs = len(alignment)
aln_len = alignment.get_alignment_length()
print(f"Alignment: {n_seqs} sequences x {aln_len} columns")

In [ ]:
# ── Per-position conservation analysis ──────────────────────────────────────
n_seqs = len(alignment)
aln_len = alignment.get_alignment_length()

conservation_scores = []   # fraction of most common residue (excluding gaps)
gap_fractions = []         # fraction of gaps at each position
consensus_residues = []    # most common residue at each position

for col_idx in range(aln_len):
    column = [str(alignment[i].seq)[col_idx] for i in range(n_seqs)]
    counts = Counter(column)

    # Gap frequency
    n_gaps = counts.get("-", 0)
    gap_frac = n_gaps / n_seqs
    gap_fractions.append(gap_frac)

    # Conservation score: frequency of most common non-gap residue
    non_gap = {k: v for k, v in counts.items() if k != "-"}
    if non_gap:
        most_common_res, most_common_count = max(non_gap.items(), key=lambda x: x[1])
        n_non_gap = sum(non_gap.values())
        cons_score = most_common_count / n_non_gap
    else:
        most_common_res = "-"
        cons_score = 0.0

    conservation_scores.append(cons_score)
    consensus_residues.append(most_common_res)

conservation_scores = np.array(conservation_scores)
gap_fractions = np.array(gap_fractions)

# Classify positions
is_conserved = conservation_scores >= CONSERVATION_THRESHOLD
is_high_gap = gap_fractions > GAP_THRESHOLD

n_conserved = is_conserved.sum()
n_nonconserved = (~is_conserved).sum()
n_high_gap = is_high_gap.sum()

print(f"Alignment length: {aln_len} columns")
print(f"Conserved positions (>={CONSERVATION_THRESHOLD*100:.0f}%): {n_conserved} ({n_conserved/aln_len*100:.1f}%)")
print(f"Non-conserved positions: {n_nonconserved} ({n_nonconserved/aln_len*100:.1f}%)")
print(f"High-gap positions (>{GAP_THRESHOLD*100:.0f}% gaps): {n_high_gap} ({n_high_gap/aln_len*100:.1f}%)")
print(f"\nMean conservation score: {conservation_scores.mean():.3f}")
print(f"Median conservation score: {np.median(conservation_scores):.3f}")

In [ ]:
# ── Conservation visualization ──────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

# 1. Conservation score across alignment
ax = axes[0]
colors = ["steelblue" if c >= CONSERVATION_THRESHOLD else "salmon" for c in conservation_scores]
ax.bar(range(aln_len), conservation_scores, color=colors, width=1.0, linewidth=0)
ax.axhline(y=CONSERVATION_THRESHOLD, color="red", linestyle="--", linewidth=0.8,
           label=f"Threshold ({CONSERVATION_THRESHOLD})")
ax.set_ylabel("Conservation\nscore")
ax.set_title("Per-position conservation across BSH alignment")
ax.legend(loc="lower right", fontsize=8)
ax.set_ylim(0, 1.05)

# 2. Gap fraction
ax = axes[1]
ax.bar(range(aln_len), gap_fractions, color="gray", width=1.0, linewidth=0)
ax.axhline(y=GAP_THRESHOLD, color="orange", linestyle="--", linewidth=0.8,
           label=f"Gap threshold ({GAP_THRESHOLD})")
ax.set_ylabel("Gap\nfraction")
ax.legend(loc="upper right", fontsize=8)
ax.set_ylim(0, 1.05)

# 3. Conservation score distribution
ax = axes[2]
ax.hist(conservation_scores, bins=50, edgecolor="black", alpha=0.7, color="steelblue")
ax.axvline(x=CONSERVATION_THRESHOLD, color="red", linestyle="--", linewidth=0.8)
ax.set_xlabel("Conservation score")
ax.set_ylabel("Count")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "conservation_plot.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nSummary:")
print(f"  Conserved (>={CONSERVATION_THRESHOLD*100:.0f}%): {n_conserved} positions")
print(f"  Non-conserved: {n_nonconserved} positions")
print(f"  High-gap (>{GAP_THRESHOLD*100:.0f}%): {n_high_gap} positions")

## Conserved Residue Analysis

The 95% conservation threshold applied to positions with ≤50% gaps reveals the **core conserved residues** shared across the BSH family. High-gap positions (>50% gaps) can appear artificially "conserved" when only a few sequences occupy them, so they are separated out below.

In [ ]:
# ── Core conserved residues: identity & biochemical properties ───────────────
# Separate truly conserved positions (low-gap) from inflated high-gap ones

cons_df_full = pd.DataFrame({
    "alignment_position": range(aln_len),
    "consensus_residue": consensus_residues,
    "conservation_score": conservation_scores,
    "gap_fraction": gap_fractions,
    "is_conserved": is_conserved,
    "is_high_gap": is_high_gap,
})

core_mask = is_conserved & ~is_high_gap
core_positions = np.where(core_mask)[0]

print(f"Conserved positions breakdown:")
print(f"  Total conserved (>={CONSERVATION_THRESHOLD*100:.0f}%):  {is_conserved.sum()}")
print(f"  High-gap conserved (>50% gaps):  {(is_conserved & is_high_gap).sum()}  <- inflated by low occupancy")
print(f"  Core conserved (<=50% gaps):     {len(core_positions)}  <- true family-wide conservation")

# Amino acid property categories
AA_PROPERTIES = {
    "G": ("Glycine",       "Nonpolar / flexible"),
    "A": ("Alanine",       "Nonpolar"),
    "V": ("Valine",        "Nonpolar"),
    "L": ("Leucine",       "Nonpolar"),
    "I": ("Isoleucine",    "Nonpolar"),
    "P": ("Proline",       "Nonpolar / rigid"),
    "F": ("Phenylalanine", "Aromatic"),
    "W": ("Tryptophan",    "Aromatic"),
    "Y": ("Tyrosine",      "Aromatic / polar"),
    "M": ("Methionine",    "Nonpolar"),
    "C": ("Cysteine",      "Polar / thiol"),
    "S": ("Serine",        "Polar"),
    "T": ("Threonine",     "Polar"),
    "N": ("Asparagine",    "Polar / amide"),
    "Q": ("Glutamine",     "Polar / amide"),
    "D": ("Aspartate",     "Negative charge"),
    "E": ("Glutamate",     "Negative charge"),
    "K": ("Lysine",        "Positive charge"),
    "R": ("Arginine",      "Positive charge"),
    "H": ("Histidine",     "Positive charge / catalytic"),
}

# Build detailed table for core conserved residues
rows = []
for pos in core_positions:
    res = consensus_residues[pos]
    col = [str(alignment[i].seq)[pos] for i in range(n_seqs)]
    counts = Counter(col)
    n_with_res = sum(v for k, v in counts.items() if k != "-")
    n_consensus = counts.get(res, 0)

    aa_name, aa_prop = AA_PROPERTIES.get(res, (res, "Unknown"))
    minority = {k: v for k, v in counts.items() if k != "-" and k != res}

    rows.append({
        "Aln. pos": pos,
        "Residue": res,
        "Name": aa_name,
        "Property": aa_prop,
        "Conservation": f"{conservation_scores[pos]:.1%}",
        "Occupancy": f"{n_with_res}/{n_seqs}",
        "Minority variants": ", ".join(f"{k}({v})" for k, v in sorted(minority.items(), key=lambda x: -x[1])) or "none",
    })

core_table = pd.DataFrame(rows)
print(f"\n{'='*90}")
print(f"  CORE CONSERVED RESIDUES IN BSH FAMILY ({len(core_positions)} positions)")
print(f"{'='*90}")
print(core_table.to_string(index=False))

# Summarize by property type
prop_counts = core_table["Property"].value_counts()
print(f"\nBy biochemical property:")
for prop, count in prop_counts.items():
    residues = core_table[core_table["Property"] == prop]["Residue"].tolist()
    print(f"  {prop}: {count} positions ({', '.join(residues)})")

In [ ]:
# ── Visualisation of conserved residues ──────────────────────────────────────
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec

# Broad property groupings for coloring
PROP_COLOR = {
    "Nonpolar / flexible": "#78c679",
    "Nonpolar / rigid":    "#31a354",
    "Nonpolar":            "#addd8e",
    "Aromatic":            "#fe9929",
    "Aromatic / polar":    "#fec44f",
    "Polar / thiol":       "#c994c7",
    "Polar":               "#d4b9da",
    "Polar / amide":       "#df65b0",
    "Negative charge":     "#ef3b2c",
    "Positive charge":     "#2171b5",
    "Positive charge / catalytic": "#6baed6",
}

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 2, height_ratios=[1.2, 1], hspace=0.35, wspace=0.3)

# ── Panel A: Core conserved residues on the alignment map ──────────────────
ax_map = fig.add_subplot(gs[0, :])

# Background: conservation score for ALL low-gap positions
low_gap_mask = gap_fractions <= GAP_THRESHOLD
low_gap_pos = np.where(low_gap_mask)[0]
low_gap_scores = conservation_scores[low_gap_mask]

ax_map.bar(low_gap_pos, low_gap_scores, color="#d9d9d9", width=1.0, linewidth=0, label="Variable")

# Overlay the core conserved positions with color by property
for _, row in core_table.iterrows():
    pos = row["Aln. pos"]
    color = PROP_COLOR.get(row["Property"], "#333333")
    ax_map.bar(pos, conservation_scores[pos], color=color, width=3.0, linewidth=0.5, edgecolor="black")
    ax_map.text(pos, conservation_scores[pos] + 0.02, row["Residue"],
                ha="center", va="bottom", fontsize=8, fontweight="bold", rotation=90)

ax_map.axhline(y=CONSERVATION_THRESHOLD, color="red", linestyle="--", linewidth=0.8, alpha=0.6)
ax_map.set_xlim(low_gap_pos.min() - 5, low_gap_pos.max() + 5)
ax_map.set_ylim(0, 1.15)
ax_map.set_xlabel("Alignment position (low-gap columns only)")
ax_map.set_ylabel("Conservation score")
ax_map.set_title("Core conserved residues across the BSH alignment\n(colored by amino acid property)")

# Legend for property types that actually appear
used_props = core_table["Property"].unique()
legend_handles = [Patch(facecolor=PROP_COLOR[p], edgecolor="black", linewidth=0.5, label=p)
                  for p in used_props if p in PROP_COLOR]
ax_map.legend(handles=legend_handles, loc="lower right", fontsize=7, ncol=2)

# ── Panel B: Residue frequency heatmap at core conserved positions ─────────
ax_heat = fig.add_subplot(gs[1, 0])

amino_acids = sorted("ACDEFGHIKLMNPQRSTVWY")
freq_matrix = np.zeros((len(amino_acids), len(core_positions)))

for j, pos in enumerate(core_positions):
    col = [str(alignment[i].seq)[pos] for i in range(n_seqs)]
    counts = Counter(col)
    n_total = sum(v for k, v in counts.items() if k != "-")
    if n_total > 0:
        for k, aa in enumerate(amino_acids):
            freq_matrix[k, j] = counts.get(aa, 0) / n_total

im = ax_heat.imshow(freq_matrix, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
ax_heat.set_xticks(range(len(core_positions)))
ax_heat.set_xticklabels([f"{consensus_residues[p]}{p}" for p in core_positions],
                         rotation=45, ha="right", fontsize=7)
ax_heat.set_yticks(range(len(amino_acids)))
ax_heat.set_yticklabels(amino_acids, fontsize=8)
ax_heat.set_xlabel("Conserved position (consensus residue + aln. position)")
ax_heat.set_ylabel("Amino acid")
ax_heat.set_title("Residue frequency at core conserved positions")
plt.colorbar(im, ax=ax_heat, label="Frequency", shrink=0.8)

# ── Panel C: Conservation tier distribution (low-gap positions only) ───────
ax_tier = fig.add_subplot(gs[1, 1])

low_gap_df = cons_df_full[cons_df_full["gap_fraction"] <= GAP_THRESHOLD].copy()
tier_bins = [0, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 1.01]
tier_labels = ["<30%", "30-50%", "50-70%", "70-80%", "80-90%", "90-95%", ">=95%"]
tier_colors = ["#d73027", "#fc8d59", "#fee08b", "#d9ef8b", "#91cf60", "#1a9850", "#004529"]
low_gap_df["tier"] = pd.cut(low_gap_df["conservation_score"], bins=tier_bins, labels=tier_labels)
tier_counts = low_gap_df["tier"].value_counts().reindex(tier_labels)

bars = ax_tier.bar(tier_labels, tier_counts.values, color=tier_colors, edgecolor="black", linewidth=0.5)
for bar, count in zip(bars, tier_counts.values):
    ax_tier.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(count), ha="center", va="bottom", fontsize=8)
ax_tier.set_xlabel("Conservation score tier")
ax_tier.set_ylabel("Number of positions")
ax_tier.set_title(f"Conservation distribution\n({len(low_gap_df)} low-gap positions)")
ax_tier.tick_params(axis="x", rotation=30)

plt.savefig(OUTPUT_DIR / "conserved_residues_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nSaved: outputs/conserved_residues_analysis.png")

## Pairwise Sequence Identity Analysis

Beyond per-position conservation, we need to know how similar the 127 BSH enzymes are **to each other**. If closely related enzymes (>90% identity) end up in different train/test splits, the model could "cheat" via data leakage. This section computes all pairwise sequence identities from the alignment and identifies clusters of highly similar enzymes.

In [ ]:
%%time
# ── Compute pairwise sequence identity from MSA ────────────────────────────
# Identity = fraction of aligned positions where both sequences have the
# same residue (excluding positions where either has a gap)

n_seqs = len(alignment)
enzyme_ids_aln = [rec.id.split("_")[-1] for rec in alignment]
seqs = [str(rec.seq) for rec in alignment]

# Pairwise identity matrix
identity_matrix = np.zeros((n_seqs, n_seqs))

for i in range(n_seqs):
    identity_matrix[i, i] = 1.0
    for j in range(i + 1, n_seqs):
        matches = 0
        compared = 0
        for k in range(len(seqs[i])):
            ri, rj = seqs[i][k], seqs[j][k]
            if ri != '-' and rj != '-':
                compared += 1
                if ri == rj:
                    matches += 1
        identity = matches / compared if compared > 0 else 0.0
        identity_matrix[i, j] = identity
        identity_matrix[j, i] = identity

# Extract upper triangle (unique pairs)
triu_idx = np.triu_indices(n_seqs, k=1)
pairwise_identities = identity_matrix[triu_idx]

print(f"=== PAIRWISE SEQUENCE IDENTITY ===")
print(f"Total unique pairs: {len(pairwise_identities):,}")
print(f"\nIdentity distribution:")
print(f"  Min:    {pairwise_identities.min():.1%}")
print(f"  Max:    {pairwise_identities.max():.1%}")
print(f"  Mean:   {pairwise_identities.mean():.1%}")
print(f"  Median: {np.median(pairwise_identities):.1%}")
print(f"  Std:    {pairwise_identities.std():.1%}")

# Count pairs above common thresholds
for thresh in [0.3, 0.5, 0.7, 0.8, 0.9, 0.95]:
    n_above = (pairwise_identities >= thresh).sum()
    print(f"  Pairs >= {thresh:.0%} identity: {n_above} ({100*n_above/len(pairwise_identities):.2f}%)")

In [ ]:
# ── Visualize pairwise identity ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Histogram of pairwise identities
ax = axes[0]
ax.hist(pairwise_identities * 100, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(30, color='green', linestyle='--', linewidth=1.5, label='30% (twilight zone)')
ax.axvline(50, color='orange', linestyle='--', linewidth=1.5, label='50%')
ax.axvline(90, color='red', linestyle='--', linewidth=1.5, label='90% (near-identical)')
ax.set_xlabel('Pairwise Sequence Identity (%)')
ax.set_ylabel('Number of pairs')
ax.set_title(f'Pairwise Identity Distribution\n(n={len(pairwise_identities):,} pairs)')
ax.legend(fontsize=8)

# 2. Identity matrix heatmap (sorted by hierarchical clustering)
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

ax = axes[1]
dist_matrix = 1 - identity_matrix
condensed = squareform(dist_matrix)
Z = linkage(condensed, method='average')
order = leaves_list(Z)
sorted_matrix = identity_matrix[np.ix_(order, order)]

im = ax.imshow(sorted_matrix, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
ax.set_title('Pairwise Identity Matrix\n(hierarchically clustered)')
ax.set_xlabel('Enzyme index')
ax.set_ylabel('Enzyme index')
plt.colorbar(im, ax=ax, label='Sequence Identity', shrink=0.8)

# 3. For each enzyme, show its max identity to any other enzyme
ax = axes[2]
max_identity_per_enzyme = []
for i in range(n_seqs):
    others = [identity_matrix[i, j] for j in range(n_seqs) if j != i]
    max_identity_per_enzyme.append(max(others))

max_identity_per_enzyme = np.array(max_identity_per_enzyme)
sorted_idx = np.argsort(max_identity_per_enzyme)[::-1]

colors_bar = ['#d62728' if v >= 0.9 else '#ff7f0e' if v >= 0.7 else 'steelblue' 
              for v in max_identity_per_enzyme[sorted_idx]]
ax.bar(range(n_seqs), max_identity_per_enzyme[sorted_idx] * 100, color=colors_bar, width=1.0)
ax.axhline(90, color='red', linestyle='--', linewidth=1, alpha=0.7, label='>90%')
ax.axhline(70, color='orange', linestyle='--', linewidth=1, alpha=0.7, label='>70%')
ax.set_xlabel('Enzyme (sorted by max identity)')
ax.set_ylabel('Max identity to any other enzyme (%)')
ax.set_title('Nearest Neighbor Identity\n(red >90%, orange >70%)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pairwise_identity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSaved: outputs/pairwise_identity_analysis.png")

In [ ]:
# ── List highly similar pairs (>90% identity) ──────────────────────────────
print("=== ENZYME PAIRS WITH >90% SEQUENCE IDENTITY ===\n")

high_id_pairs = []
for i in range(n_seqs):
    for j in range(i + 1, n_seqs):
        if identity_matrix[i, j] >= 0.90:
            high_id_pairs.append({
                'Enzyme_1': enzyme_ids_aln[i],
                'Enzyme_2': enzyme_ids_aln[j],
                'Identity': identity_matrix[i, j],
                'Full_ID_1': alignment[i].id,
                'Full_ID_2': alignment[j].id,
            })

if high_id_pairs:
    df_high = pd.DataFrame(high_id_pairs).sort_values('Identity', ascending=False)
    print(f"Found {len(df_high)} pairs with >90% identity:\n")
    for _, row in df_high.iterrows():
        print(f"  {row['Full_ID_1']:30s} vs {row['Full_ID_2']:30s}  → {row['Identity']:.1%}")
    
    # Which enzymes appear most often in high-identity pairs?
    all_enzymes_high = list(df_high['Full_ID_1']) + list(df_high['Full_ID_2'])
    from collections import Counter as Ctr
    enzyme_freq = Ctr(all_enzymes_high)
    print(f"\nEnzymes involved in >90% identity pairs:")
    for enz, count in enzyme_freq.most_common():
        print(f"  {enz}: {count} pair(s)")
    
    # Save
    df_high.to_csv(OUTPUT_DIR / 'high_identity_pairs.csv', index=False)
    print(f"\nSaved: {OUTPUT_DIR / 'high_identity_pairs.csv'}")
else:
    print("No pairs found with >90% identity — the dataset is well-diversified!")

# Also check >80%
n_above_80 = ((pairwise_identities >= 0.80) & (pairwise_identities < 0.90)).sum()
print(f"\nPairs at 80-90% identity: {n_above_80}")
print(f"Pairs at 70-80% identity: {((pairwise_identities >= 0.70) & (pairwise_identities < 0.80)).sum()}")

In [ ]:
# ── Per-sequence non-conserved residue mapping ──────────────────────────────
# For each sequence, map alignment columns back to original residue indices.
# Only keep non-conserved alignment positions where the sequence has an
# actual residue (not a gap).

nonconserved_cols = set(np.where(~is_conserved)[0])

residue_mapping = {}  # enzyme_id -> list of original (0-indexed) residue positions

for i, record in enumerate(alignment):
    enzyme_id = record.id.split("_")[-1]
    seq = str(record.seq)

    original_idx = -1  # will be incremented to 0 for first residue
    nonconserved_positions = []

    for col_idx, residue in enumerate(seq):
        if residue != "-":
            original_idx += 1
            if col_idx in nonconserved_cols:
                nonconserved_positions.append(original_idx)

    residue_mapping[enzyme_id] = nonconserved_positions

# Summary
n_noncons_per_seq = [len(v) for v in residue_mapping.values()]
print(f"Non-conserved residue count per sequence:")
print(f"  Min: {min(n_noncons_per_seq)}")
print(f"  Max: {max(n_noncons_per_seq)}")
print(f"  Mean: {np.mean(n_noncons_per_seq):.1f}")
print(f"  Median: {np.median(n_noncons_per_seq):.0f}")

# Show example
example_id = list(residue_mapping.keys())[0]
print(f"\nExample: {example_id} has {len(residue_mapping[example_id])} non-conserved residues")
print(f"  First 20 positions (0-indexed): {residue_mapping[example_id][:20]}")

In [ ]:
# ── Export results ──────────────────────────────────────────────────────────

# 1. Conservation scores per alignment position
cons_df = pd.DataFrame({
    "alignment_position": range(aln_len),
    "consensus_residue": consensus_residues,
    "conservation_score": conservation_scores,
    "gap_fraction": gap_fractions,
    "is_conserved": is_conserved,
    "is_high_gap": is_high_gap,
})
cons_csv = OUTPUT_DIR / "conservation_scores.csv"
cons_df.to_csv(cons_csv, index=False)
print(f"Saved conservation scores: {cons_csv}")
print(f"  {len(cons_df)} rows (alignment positions)")
print(cons_df.head())

# 2. Non-conserved residue indices per enzyme
noncons_rows = []
for enzyme_id, positions in residue_mapping.items():
    noncons_rows.append({
        "enzyme_id": enzyme_id,
        "n_nonconserved": len(positions),
        "nonconserved_positions": ";".join(map(str, positions)),
    })
noncons_df = pd.DataFrame(noncons_rows)
noncons_csv = OUTPUT_DIR / "nonconserved_residue_indices.csv"
noncons_df.to_csv(noncons_csv, index=False)
print(f"\nSaved non-conserved indices: {noncons_csv}")
print(f"  {len(noncons_df)} rows (enzymes)")
print(noncons_df.head())

## Notes on Future Per-Residue Embedding Extraction

The outputs from this notebook enable a targeted embedding strategy:

1. **Full sequence** -> pass through ProtT5 (or ESM-2) to get **per-residue embeddings** (shape: `[L, d]` where `L` = sequence length, `d` = embedding dimension)
2. **Select non-conserved positions** -> use `nonconserved_residue_indices.csv` to index only the residues at non-conserved alignment positions for each enzyme
3. **Mean-pool** the selected per-residue embeddings -> produces a single vector per enzyme that captures only the variable/distinguishing regions

This approach filters out the conserved "structural backbone" signal shared across all BSH enzymes, focusing the representation on the positions most likely to explain functional differences (e.g., substrate specificity).

### File references
- `outputs/conservation_scores.csv` — per alignment position: conservation score, gap fraction, conserved/non-conserved label
- `outputs/nonconserved_residue_indices.csv` — per enzyme: list of original (pre-alignment) residue indices that are non-conserved
- `outputs/bsh_aligned.fasta` — the full multiple sequence alignment